# 14 — G-Eval (LLM-as-a-Judge, backfill sulla split test)

Aggiunge al benchmark una metrica **LLM-as-a-Judge**: G-Eval (Liu et al., 2023) fa valutare a
un modello linguistico ogni riassunto generato su quattro dimensioni, in scala **1-5**:

| dimensione | cosa misura |
|---|---|
| **coherence** | il riassunto e' ben strutturato e si legge come un testo unico |
| **consistency** | fedelta' fattuale rispetto alla sorgente (allucinazioni, contraddizioni) |
| **fluency** | correttezza grammaticale e scorrevolezza |
| **relevance** | quanta informazione importante della sorgente viene catturata |

E' l'unica metrica del benchmark che non passa dal riferimento umano: ROUGE/BLEU/METEOR
misurano **sovrapposizione lessicale** con il `summary` di riferimento e BERTScore
**similarita' semantica** con lo stesso. Nessuna delle due dice se un riassunto e' coerente o
se ha inventato dei fatti — cioe' proprio le dimensioni su cui i sistemi astrattivi
differiscono davvero.

## Perche' non `psr.g_eval()`

`pyAutoSummarizer` (gia' usato per ROUGE/BLEU/METEOR, vedi `summ_utils.crea_valutatore`)
espone gia' un G-Eval. Ispezionandone il codice (`pyAutoSummarizer/base/psr.py`, v1.2.0)
risulta inutilizzabile qui per **quattro** motivi:

1. costruisce `openai.OpenAI(api_key=...)` **senza `base_url`**: parla solo con OpenAI, non
   con Azure (ne' con l'endpoint ollama locale usato dai notebook 07-09);
2. ha `max_tokens=5` e `temperature=0.0` **cablati**, entrambi fatali per un modello
   reasoning (che rifiuta `temperature` e spende il budget in token di ragionamento);
3. legge la sorgente da `self.full_txt`, cioe' dal testo passato al costruttore: servirebbe
   una `psr.summarization(sorgente)` nuova — e pesante — per **ogni** esempio, e il pattern a
   istanza fittizia condivisa di `crea_valutatore()` non regge;
4. fa **una chiamata API per dimensione**, quadruplicando il costo.

Qui la funzione viene quindi reimplementata in `summ_utils.py`, ma le **rubriche restano
verbatim**: `su.RUBRICHE_GEVAL` e' derivato meccanicamente da `su.PROMPT_GEVAL_ORIGINALI`
(copia letterale dei template della libreria) tagliando la coda `"Reply with a single digit
only."`, qui sostituita dalla richiesta di un unico oggetto JSON. La metrica resta cosi'
difendibile come *il G-Eval di pyAutoSummarizer, reimplementato per un endpoint non-OpenAI*.

## Scelta del giudice: `gpt-5.4-mini` su Azure AI Foundry

Deployment **GlobalStandard** dedicato su `agirasella-resource` (swedencentral). Due proprieta'
lo motivano:

1. **Indipendente da tutti e 13 i metodi valutati**: nessun self-judging. Usare `gpt-5-mini`
   (il modello del notebook 12, presente nel benchmark come metodo `gpt5mini`) significherebbe
   farlo giudicare i propri output, con il noto bias di auto-preferenza.
2. **Piu' recente di ogni generatore del benchmark** (`gpt-5-mini` e' 2025-08-07,
   `gpt-5.4-mini` e' 2026-03-17). Conta soprattutto per **consistency**: individuare
   un'allucinazione sottile dentro un cluster multi-documento e' un compito di ragionamento, e
   la correlazione giudice-umano scala con la capacita' del giudice proprio su quella
   dimensione. Un giudice piu' debole tenderebbe a schiacciare i punteggi verso l'alto,
   penalizzando poco i modelli forti.

Verificato dal vivo sulla sottoscrizione: DeepSeek/Grok/Llama/Mistral **non sono deployabili**
su questo account AIServices (stessa ragione per cui i notebook Claude Haiku e DeepSeek furono
abbandonati), e **`gpt-5.1-mini` non esiste** — la linea 5.1 e' `gpt-5.1` / `-chat` /
`-codex*`.

### Conseguenza: e' un modello reasoning

Valgono le stesse regole del notebook 12: **niente `temperature`** (accetta solo il default),
`max_completion_tokens` al posto di `max_tokens`, `reasoning_effort='minimal'`. I token di
ragionamento consumano il budget **prima** dell'output visibile: e' lo stesso modo di fallire
gia' visto con gemma (notebook 08) e gpt-5-mini (notebook 12), per questo il budget e' 1500 e
non 200. Le corse **non sono riproducibili bit-a-bit**: l'artefatto di riproducibilita' e' la
cache JSONL committata, da cui le metriche si riderivano a costo zero.

## Protocollo

- **una chiamata per (metodo, riga)**, con tutte e quattro le dimensioni in un unico JSON
  (`response_format` con `json_schema` strict: il formato e' garantito lato server);
- sorgente troncata a **2.100 parole** (`su.MAX_PAROLE_SORGENTE_GEVAL`), sopra le 1.024 token
  sotto cui la prompt cache di Azure non si attiva e comunque piu' di quanto vedano
  BART/PEGASUS (1.024 token);
- ambito: **split test intera**, 13 metodi x 5.610 righe = **72.930 giudizi**;
- tier standard concorrente (niente Batch API: `gpt-5.4-mini` la supporta, ma il ritardo di
  24 h per job e i limiti di 200 MB/file la rendono piu' scomoda della prompt cache).

## Ambito e output

**Backfill una tantum**, come il notebook 13: non tocca i notebook 01-04/06-12 ne' il loro
ciclo di generazione. I 13 metodi sono quelli della Vista 2 del notebook 05.

I punteggi vanno in **file separati** (`{metodo}_test_geval_per_example.csv` e
`..._geval_aggregate.json`), **mai** uniti al CSV per-esempio standard. Il motivo e' in
`summ_utils.valuta_e_salva`, la cui media interna somma **ogni** colonna su **ogni** riga: il
giudice lascia scoperte alcune righe (content filter di Azure, parsing fallito) e la media
esploderebbe con un `KeyError`; e restringere le righe valutate riscriverebbe i 13 CSV gia'
committati, cambiando medie e `n_esempi` di ROUGE/BLEU/METEOR/BERTScore.

In [ ]:
# --- Dipendenze --------------------------------------------------------------
# Su Colab la libreria openai potrebbe non esserci: installazione idempotente.
try:
    import openai  # noqa: F401
except ImportError:
    %pip install -q openai
    import openai  # noqa: F401

In [ ]:
# --- Configurazione ----------------------------------------------------------
import json
import os
import random
import time
from collections import Counter

import summ_utils as su

SCOPE = os.environ.get('GEVAL_SCOPE', 'test')

GIUDICE = 'gpt-5.4-mini'
# Su Azure OpenAI `model=` e' il nome del DEPLOYMENT, non del modello
DEPLOYMENT_GIUDICE = os.environ.get('AZURE_GEVAL_DEPLOYMENT', 'gpt-5.4-mini')
AZURE_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']   # radice della risorsa, senza path
AZURE_API_KEY = os.environ['AZURE_OPENAI_API_KEY']
# API "v1" di Azure OpenAI: nessuna api-version datata (come nel notebook 12)
BASE_URL = AZURE_ENDPOINT.rstrip('/') + '/openai/v1/'

# Parametri del giudice. gpt-5.4-mini e' un modello REASONING:
#  - `temperature` non e' accettata (solo il default)
#  - i token di reasoning consumano il budget PRIMA dell'output visibile, quindi
#    1500 e non 200 (stesso valore gia' validato nel notebook 12)
MAX_COMPLETION_TOKENS = 1500
REASONING_EFFORT = 'minimal'
MAX_PAROLE = su.MAX_PAROLE_SORGENTE_GEVAL   # 2100

# Concorrenza e controllo di spesa (sovrascrivibili da scripts/run_geval.py)
N_THREAD = int(os.environ.get('GEVAL_THREAD', 8))
LIMIT_RIGHE = int(os.environ['GEVAL_RIGHE']) if os.environ.get('GEVAL_RIGHE') else None
BUDGET_MASSIMO = float(os.environ['GEVAL_BUDGET']) if os.environ.get('GEVAL_BUDGET') else None
RIPROVA_ERRORI = os.environ.get('GEVAL_RIPROVA_ERRORI') == '1'
SOLO_METRICHE = os.environ.get('GEVAL_SOLO_METRICHE') == '1'
PILOTA = int(os.environ['GEVAL_PILOTA']) if os.environ.get('GEVAL_PILOTA') else 0
OGNI = int(os.environ.get('GEVAL_OGNI', 1500))   # cadenza del blocco costo

BASE = su.trova_base_dir()
P = su.percorsi_standard(BASE)

# Stessi 13 metodi con metriche 'test' del notebook 05 (Vista 2) e del notebook 13
METODI_BASELINE = ['firstk_psr', 'firstk_nltk',          # notebook 10 (First-k)
                   'centroid_mmr', 'centroid_mmr_bert']  # notebook 11 (Centroid+MMR)
METODI = METODI_BASELINE + ['textrank', 'lexrank', 'bart', 'pegasus', 'primera',
                            'qwen', 'gemma', 'mistral'] + ['gpt5mini']


def percorso_riassunti(metodo):
    """TextRank/LexRank non hanno una corsa '_test.tsv' dedicata: i riassunti sono
    nel file '_full.tsv' (intero complete.tab). I riferimenti sotto sono comunque gia'
    ristretti alla sola split test, quindi il filtro avviene automaticamente."""
    suffisso = 'full' if metodo in ('textrank', 'lexrank') else 'test'
    return P['summaries_dir'] / f'{metodo}_{suffisso}.tsv'


CACHE_PATH = P['metrics_dir'] / f'geval_cache_{SCOPE}.jsonl'

# Prezzi unitari dal listino Azure (fallback: su.PREZZI_GEVAL). Non esiste un'API
# Azure di costo in tempo reale: il costo si calcola dai token dell'oggetto `usage`.
PREZZI = su.prezzi_retail_azure()

config = {
    'giudice': GIUDICE,
    'deployment': DEPLOYMENT_GIUDICE,
    'backend': 'Azure AI Foundry / Azure OpenAI (chat completions, rotta v1)',
    'max_completion_tokens': MAX_COMPLETION_TOKENS,
    'reasoning_effort': REASONING_EFFORT,
    'temperature': 'non impostata (modello reasoning: accetta solo il default)',
    'max_parole_sorgente': MAX_PAROLE,
    'protocollo': 'una chiamata per (metodo, riga), 4 dimensioni in un unico JSON',
    'response_format': 'json_schema strict',
    'rubriche': 'verbatim da pyAutoSummarizer 1.2.0 (psr.summarization.g_eval)',
    'n_thread': N_THREAD,
    'prezzi_usd_per_1M_token': PREZZI,
    'nota': ('punteggi 1-5 su coherence/consistency/fluency/relevance; il giudice e\' '
             'indipendente da tutti i metodi valutati e piu\' recente di ogni generatore '
             'del benchmark'),
}

print(f'Scope        : {SCOPE}')
print(f'Giudice      : {GIUDICE} (deployment {DEPLOYMENT_GIUDICE})')
print(f'Cache        : {CACHE_PATH}')
print(f'Thread       : {N_THREAD}   report ogni {OGNI} giudizi')
print(f'Budget max   : {("$%.2f" % BUDGET_MASSIMO) if BUDGET_MASSIMO else "nessuno"}')
print(f'Limite righe : {LIMIT_RIGHE or "tutte"}   pilota: {PILOTA or "no"}')

## Prompt e formato della risposta

Due vincoli guidano la costruzione dei messaggi, entrambi implementati in
`su.costruisci_messaggi_geval`:

**1. L'ordine e' un contratto, non uno stile.** Il messaggio `system` (le quattro rubriche +
la richiesta del JSON) e' **identico su tutte le 72.930 chiamate**, e il messaggio `user`
comincia con la **sorgente troncata** — la stessa per tutti e 13 i metodi di una data riga —
mettendo in fondo il **riassunto**, unica parte che cambia:

```
system : <rubriche + formato JSON>          <- costante
user   : Source:\n<sorgente troncata>       <- costante dentro la riga
         \n\nSummary:\n<riassunto>          <- varia per metodo
```

Cosi' il **primo** dei 13 giudizi di una riga popola la prompt cache di Azure e gli altri
dodici la riusano (~92% del prefisso in cache, pagato a tariffa ridotta). Invertire l'ordine —
riassunto prima della sorgente — azzererebbe il risparmio: e' la differenza fra ~$39 e ~$180
di soli token di input sulla corsa completa.

**2. Il formato lo garantisce il server.** `response_format` con `json_schema` in modalita'
`strict` (`su.SCHEMA_GEVAL`) obbliga il modello a produrre esattamente quattro interi. Nota:
in `strict` non sono ammessi `minimum`/`maximum`, quindi la scala 1-5 e' espressa con `enum`.
`su.estrai_punteggi_geval` resta comunque difensivo (code fence, testo attorno, chiavi
mancanti, valori fuori scala) e **solleva** invece di inventare un punteggio.

In [ ]:
# --- Client Azure e singolo giudizio -----------------------------------------
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=AZURE_API_KEY)

print(su.PROMPT_GEVAL_SISTEMA)
print('\n' + '-' * 70)
print('Schema JSON:', json.dumps(su.SCHEMA_GEVAL))


def giudica_uno(sorgente, riassunto):
    """Un giudizio G-Eval -> (punteggi, usage). Solleva su risposta vuota o non conforme."""
    def chiamata():
        return client.chat.completions.create(
            model=DEPLOYMENT_GIUDICE,
            messages=su.costruisci_messaggi_geval(sorgente, riassunto, MAX_PAROLE),
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            reasoning_effort=REASONING_EFFORT,
            response_format={'type': 'json_schema',
                             'json_schema': {'name': 'geval', 'strict': True,
                                             'schema': su.SCHEMA_GEVAL}})

    risposta = su.chiama_con_backoff(chiamata)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # budget di reasoning esaurito prima dell'output visibile: stesso modo di
        # fallire di gemma (nb 08) e gpt-5-mini (nb 12). Si solleva -> nessun
        # punteggio inventato, la riga resta ritentabile.
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return su.estrai_punteggi_geval(contenuto), risposta.usage

## Preparazione dei compiti

Riferimenti e riassunti si caricano **una sola volta** (~5.610 righe + 13 dizionari), poi si
costruisce la lista dei compiti **ordinata per `row_id`, con i 13 metodi raggruppati**.

Questo raggruppamento non e' cosmetico: `su.giudica_geval_concorrente` prende la **riga** come
unita' di lavoro ed esegue i suoi 13 giudizi **in sequenza dentro lo stesso thread**, cosi' che
il primo scaldi la prompt cache e gli altri la colpiscano. Il parallelismo e' **tra** righe
diverse. Parallelizzare per singolo giudizio farebbe partire insieme le 13 chiamate della
stessa riga, mancando la cache tutte quante.

In [ ]:
# --- Costruzione dei compiti -------------------------------------------------
riferimenti_test = list(su.itera_split(P['complete_tab'], SCOPE))
print(f'Riferimenti split {SCOPE}: {len(riferimenti_test)} righe')

riassunti = {}
for metodo in METODI:
    path = percorso_riassunti(metodo)
    riassunti[metodo] = su.carica_riassunti(path)
    if not riassunti[metodo]:
        print(f'  ATTENZIONE {metodo}: {path.name} non trovato o vuoto')

righe = []
for rif in riferimenti_test:
    rid = rif['row_id']
    coppie = [(m, riassunti[m][rid]) for m in METODI if rid in riassunti[m]]
    if coppie:
        righe.append((rid, su.prepara_documento(rif['document']), coppie))

totale_giudizi = sum(len(c) for _, _, c in righe)
print(f'\nRighe con almeno un riassunto: {len(righe)}')
print(f'Giudizi totali attesi       : {totale_giudizi:,}')
print('\nCopertura per metodo:')
for metodo in METODI:
    print(f'  {metodo:20s} {len(riassunti[metodo]):5d}')

lunghezze = [len(s.split()) for _, s, _ in righe]
tagliate = sum(1 for n in lunghezze if n > MAX_PAROLE)
print(f'\nLunghezza sorgente (parole): mediana {sorted(lunghezze)[len(lunghezze)//2]}, '
      f'max {max(lunghezze):,}')
print(f'Righe troncate a {MAX_PAROLE} parole: {tagliate} / {len(righe)} '
      f'({tagliate/len(righe):.1%})')

## Pilota

La corsa completa e' ~72.930 chiamate **a pagamento** e molte ore. Il pilota valida prompt,
parsing, cache e — soprattutto — il **costo empirico per giudizio** prima di impegnarsi.

Il campione e' costruito con seed fisso **piu' le due righe PEGASUS patologiche note**
(`51178`, ciclo di ripetizione del beam search; `56099`, probabile disallineamento
sorgente/riassunto — vedi le avvertenze sul METEOR nel README). Servono da controllo di
**discriminazione**: se G-Eval non assegna a quelle fluency/coherence bassissime, la metrica
non sta misurando nulla di utile.

Criteri di accettazione prima di lanciare la corsa completa:

- >= 95% di giudizi riusciti e **nessuna (o quasi) risposta vuota** — se il budget di reasoning
  si esaurisce, alzare `MAX_COMPLETION_TOKENS` **prima** della corsa lunga, non durante;
- `cached_tokens > 0` sui giudizi successivi al primo di ogni riga (prova che la prompt cache
  funziona: se e' 0, fermarsi e rivedere l'ordine dei messaggi);
- distribuzione **non degenere** (deviazione standard > 0 su ogni dimensione: un giudice che
  risponde sempre 4 e' inutile);
- ordinamento plausibile dei metodi (LLM astrattivi sopra le baseline posizionali);
- costo/giudizio misurato, estrapolato ai 72.930 giudizi: e' il numero **go/no-go**.

In [ ]:
# --- Pilota ------------------------------------------------------------------
RIGHE_PATOLOGICHE = [51178, 56099]   # PEGASUS: ripetizione beam search / mismatch

if PILOTA:
    per_id = {rid: r for r in righe for rid in [r[0]]}
    scelte = [per_id[rid] for rid in RIGHE_PATOLOGICHE if rid in per_id]
    resto = [r for r in righe if r[0] not in RIGHE_PATOLOGICHE]
    random.Random(42).shuffle(resto)
    righe_pilota = scelte + resto[:max(PILOTA - len(scelte), 0)]

    cache = su.CacheGiudizi(CACHE_PATH)
    riepilogo = su.giudica_geval_concorrente(
        righe_pilota, giudica_uno, cache, n_thread=N_THREAD,
        etichetta='pilota ', ogni=max(OGNI // 20, 20), prezzi=PREZZI)
    cache.chiudi()

    rimanenti = totale_giudizi - riepilogo['fatti'] - riepilogo['saltati']
    costo_giudizio = riepilogo['costo_per_giudizio']
    print('\n=== ESTRAPOLAZIONE ALLA CORSA COMPLETA ===')
    print(f"  giudizi totali        : {totale_giudizi:,}")
    print(f"  costo/giudizio misurato: ${costo_giudizio:.5f}")
    print(f"  COSTO STIMATO TOTALE  : ${costo_giudizio * totale_giudizi:.2f}")
    print(f"  ore stimate           : "
          f"{totale_giudizi / max(riepilogo['giudizi_al_s'], 1e-9) / 3600:.1f} h "
          f"(a {N_THREAD} thread)")
    print(f"  token reasoning/giudizio: "
          f"{riepilogo['reasoning_tokens'] / max(riepilogo['giudizi'], 1):.0f}")
    print(f"  quota input in cache   : {riepilogo['quota_cached']:.1%}"
          f"   <- se e' ~0, l'ordine del prompt e' sbagliato")
else:
    print('Pilota non richiesto (GEVAL_PILOTA non impostata).')

In [ ]:
# --- Controllo di discriminazione sulle righe patologiche --------------------
if PILOTA:
    cache = su.CacheGiudizi(CACHE_PATH)
    for rid in RIGHE_PATOLOGICHE:
        voci = {m: cache.per_metodo(m).get(rid) for m in METODI}
        if not any(voci.values()):
            continue
        print(f'\nrow_id={rid}')
        for metodo, v in voci.items():
            if v:
                print(f"  {metodo:20s} coh={v['geval_coherence']} "
                      f"cons={v['geval_consistency']} flu={v['geval_fluency']} "
                      f"rel={v['geval_relevance']}")
    print('\nAtteso: PEGASUS con fluency/coherence molto basse (1-2) su queste righe.')

    # distribuzione dei punteggi: il giudice deve discriminare, non rispondere sempre 4
    import statistics
    tutti = [v for m in METODI for v in cache.per_metodo(m).values()]
    if tutti:
        print(f'\nDistribuzione su {len(tutti)} giudizi:')
        for col in su.COLONNE_METRICHE_GEVAL:
            valori = [v[col] for v in tutti]
            dev = statistics.pstdev(valori)
            stato = 'OK' if dev > 0 else 'DEGENERE'
            print(f'  {col:20s} media {statistics.mean(valori):.2f}  '
                  f'dev.std {dev:.2f}  [{stato}]')
    cache.chiudi()

## Corsa completa

Riprendibile: ogni giudizio viene scritto e flushato sulla cache JSONL appena arriva, e un
rilancio salta le coppie `(metodo, row_id)` gia' presenti. Interrompere con Ctrl-C non perde
nulla di pagato.

**Monitoraggio della spesa.** Non esiste un'API Azure che riporti il costo in tempo reale:
Cost Management ha 8-24 h di ritardo. La fonte di verita' e' l'oggetto `usage` di ogni
risposta, che qui viene sommato da `su.ContatoreCosti` e stampato ogni `OGNI` giudizi
(default 1500) con: ritmo, TPM osservato, ETA, quota di input in cache, quota di token di
reasoning, costo diviso per voce e **proiezione a fine corsa**. Poiche' i conteggi finiscono
anche nella cache, la stessa stima si puo' rileggere **da un secondo terminale a corsa in
corso**, senza toccare l'API:

```
python scripts/run_geval.py --costo
```

`BUDGET_MASSIMO` (`--budget`) e' un tetto duro: al superamento la corsa si ferma in modo
pulito e basta rilanciare con un tetto piu' alto.

**Attenzione**: cambiare deployment, rubriche o troncamento **dopo** aver popolato la cache
mescolerebbe corse diverse. In quel caso cancellare prima `geval_cache_{SCOPE}.jsonl`.

In [ ]:
# --- Corsa completa ----------------------------------------------------------
if not PILOTA and not SOLO_METRICHE:
    cache = su.CacheGiudizi(CACHE_PATH)
    print(f'Giudizi gia\' in cache: {len(cache):,} / {totale_giudizi:,}')
    if RIPROVA_ERRORI:
        rimasti = cache.dimentica_errori()
        print(f'Errori rimossi dalla cache: ora {rimasti:,} voci (verranno ritentati)')

    riepilogo = su.giudica_geval_concorrente(
        righe[:LIMIT_RIGHE], giudica_uno, cache, n_thread=N_THREAD,
        etichetta='', ogni=OGNI, prezzi=PREZZI, budget_massimo=BUDGET_MASSIMO)
    cache.chiudi()

    print('\n' + json.dumps({k: v for k, v in riepilogo.items()
                             if k not in ('costo',)}, indent=2, default=str))
else:
    print('Corsa completa saltata (modalita\' pilota o solo-metriche).')

## Scrittura delle metriche

Passo **separato e a costo zero**: rilegge solo la cache, senza alcuna chiamata API — lo stesso
principio del resto del repository, dove la valutazione legge esclusivamente i file salvati.
Puo' quindi essere ripetuto quante volte serve (`scripts/run_geval.py --solo-metriche`), anche
a corsa non finita, per avere una fotografia parziale.

La `config` storica di ciascun metodo **non** viene toccata: questi sono file nuovi, separati
da `{metodo}_{scope}_aggregate.json`, e la `config` che ci scriviamo dentro e' quella del
**giudice**, non quella della corsa di generazione.

In [ ]:
# --- Scrittura dei file di metrica -------------------------------------------
cache = su.CacheGiudizi(CACHE_PATH)
errori_per_metodo = Counter(m for m, _, _ in cache.errori())

riepilogo_metodi = {}
for metodo in METODI:
    punteggi = cache.per_metodo(metodo)
    if not punteggi:
        print(f'({metodo}: nessun giudizio in cache, salto)')
        continue
    n_attesi = sum(1 for _, _, coppie in righe if metodo in dict(coppie))
    diagnostica = {
        'n_giudizi_riusciti': len(punteggi),
        'n_richiesti': n_attesi,
        'n_errori': errori_per_metodo.get(metodo, 0),
        'copertura': round(len(punteggi) / n_attesi, 4) if n_attesi else 0.0,
    }
    _, aggregato = su.salva_metriche_geval(punteggi, metodo, SCOPE, P['metrics_dir'],
                                           config, diagnostica=diagnostica)
    riepilogo_metodi[metodo] = aggregato

print('\nG-Eval medio per metodo (scala 1-5):')
print(f"{'metodo':22s} {'n':>6s} {'coher':>6s} {'consi':>6s} {'fluen':>6s} "
      f"{'relev':>6s} {'MEDIA':>6s}")
for metodo, agg in sorted(riepilogo_metodi.items(),
                          key=lambda kv: -kv[1]['overall']['geval_media']):
    o = agg['overall']
    print(f"{metodo:22s} {agg['n_esempi']:6d} {o['geval_coherence']:6.2f} "
          f"{o['geval_consistency']:6.2f} {o['geval_fluency']:6.2f} "
          f"{o['geval_relevance']:6.2f} {o['geval_media']:6.2f}")

## Diagnostica e meta-valutazione

Tre controlli, nell'ordine in cui contano:

1. **il giudice discrimina?** Se la deviazione standard di una dimensione e' ~0, il modello sta
   rispondendo sempre lo stesso numero e quella dimensione va buttata.
2. **cosa e' andato storto?** Errori raggruppati per tipo. Le righe respinte dal content filter
   di Azure sono deterministiche (le stesse 139 che mancano a `gpt5mini` nel notebook 12) e non
   sono ritentabili senza un filtro custom *high-only* attaccato al deployment.
3. **G-Eval dice qualcosa di nuovo?** La correlazione di Spearman fra `geval_media` e
   `rouge1_f1`/`bertscore_f1` e' il vero contributo: se fosse ~1, la metrica sarebbe ridondante;
   dove diverge, sta misurando qualcosa che le metriche lessicali non vedono.

In [ ]:
# --- Diagnostica -------------------------------------------------------------
import statistics

import pandas as pd

tutti = [v for m in METODI for v in cache.per_metodo(m).values()]
print(f'Distribuzione dei punteggi su {len(tutti):,} giudizi:')
for col in su.COLONNE_METRICHE_GEVAL:
    valori = [v[col] for v in tutti]
    dev = statistics.pstdev(valori)
    print(f'  {col:20s} media {statistics.mean(valori):.2f}  dev.std {dev:.2f}  '
          f'[{"OK" if dev > 0.05 else "SOSPETTO: giudice poco discriminante"}]')

errori = cache.errori()
print(f'\nGiudizi falliti in cache: {len(errori):,}')
if errori:
    tipi = Counter('content_filter' if 'content_filter' in e.lower() else e[:60]
                   for _, _, e in errori)
    for tipo, n in tipi.most_common(10):
        print(f'  {n:6d}  {tipo}')

# Correlazione con le metriche esistenti
print('\nCorrelazione di Spearman (per metodo, sulle righe in comune):')
print(f"{'metodo':22s} {'n':>6s} {'vs ROUGE-1':>11s} {'vs BERTScore':>13s}")
for metodo in METODI:
    geval_path = P['metrics_dir'] / f'{metodo}_{SCOPE}_geval_per_example.csv'
    std_path = P['metrics_dir'] / f'{metodo}_{SCOPE}_per_example.csv'
    if not (geval_path.exists() and std_path.exists()):
        continue
    df = pd.read_csv(std_path).merge(pd.read_csv(geval_path), on='row_id', how='inner')
    if len(df) < 50:
        continue
    r_rouge = df['geval_media'].corr(df['rouge1_f1'], method='spearman')
    r_bert = (df['geval_media'].corr(df['bertscore_f1'], method='spearman')
              if 'bertscore_f1' in df.columns else float('nan'))
    print(f'{metodo:22s} {len(df):6d} {r_rouge:11.3f} {r_bert:13.3f}')

# Ispezione qualitativa: i giudizi piu' bassi
print('\n10 giudizi piu\' bassi (ispezione qualitativa):')
peggiori = sorted(((v['geval_media'], m, rid)
                   for m in METODI for rid, v in cache.per_metodo(m).items()))[:10]
for media, metodo, rid in peggiori:
    testo = riassunti[metodo].get(rid, '')
    print(f"\n  [{media:.2f}] {metodo} row_id={rid}")
    print(f"  {testo[:300]}")
cache.chiudi()

## Validazione consigliata prima della corsa completa

Scala a tre gradini, dal piu' economico (`scripts/run_geval.py` imposta le variabili
d'ambiente per conto suo):

1. **smoke a 1 riga** — `python scripts/run_geval.py --righe 1` (13 chiamate). Verifica
   endpoint, deployment, chiave, conformita' del JSON, assenza di risposte vuote e soprattutto
   che `cached_tokens > 0` sulle chiamate successive alla prima: e' la prova empirica che la
   prompt cache funziona. Se e' 0, **fermarsi** e rivedere l'ordine dei messaggi prima di
   spendere altro.
2. **pilota a 20 righe** — `python scripts/run_geval.py --pilota 20` (260 giudizi). Restituisce
   il costo/giudizio misurato ed estrapolato ai 72.930, i token di reasoning effettivi, il TPM
   osservato e i criteri di accettazione elencati sopra.
3. **corsa completa** — `python scripts/run_geval.py --budget <tetto>`, riprendibile.

Dopo la corsa completa, ri-eseguire il **notebook 05** per aggiornare le viste di confronto con
le nuove colonne G-Eval, e verificare con `git status` che i 13 CSV per-esempio **standard**
risultino immutati: e' il controllo diretto sul fatto che il backfill non abbia toccato le
metriche gia' pubblicate.